### 階段一：初始化與資料收集
### 1-1 套件安裝


In [1]:
import subprocess, sys
for pkg in ['statsmodels','plotly','ipywidgets','kaleido','yfinance','joblib','dtaidistance','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import warnings; warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
from statsmodels.tsa.stattools import coint
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML
pd.set_option('display.float_format','{:.4f}'.format)
import yfinance as yf
import time
# print('套件導入完成。')

In [2]:
import os
import math

# ==========================================
# 🛑 全局模式切換開關 (True: 快速開發測試 / False: 論文最終完整回測)
# ==========================================
FAST_TEST_MODE = True

if FAST_TEST_MODE:
    print("【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = 'Information Technology'  
else:
    print("【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# ==========================================
# 📊 共用策略參數 (兩種模式皆適用)
# ==========================================
DB_PATH           = r'..\data\sp500.db'
# 產業動態補齊開關與快取路徑
USE_DYNAMIC_SECTORS = True      # 啟用動態產業補齊（True: 執行補齊邏輯 / False: 僅使用原始資料庫的產業分類）
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'
SECTOR_NEUTRAL    = True        # 是否執行產業中性化（True: 僅配對同產業股票 / False: 不限制產業配對）

# 視窗滾動參數
FORMATION_WINDOW = 252   # 形成期 (約一年)
TRADING_WINDOW = 126     # 交易期 (約半年)
ROLLING_WINDOW = 20      # 滾動步長 (約一個月)
MIN_HISTORY_DAYS  = 200  # 最少歷史資料

# 配對與統計檢定參數
MAX_PAIRS_PER_TRANCHE = 1 # 決定每個梯隊要執行的最大配對數 
TOP_N_PAIRS       = 5     # 每個梯隊選擇前 N 名配對進行交易
COINT_P_VALUE     = 0.01  # 嚴格的共整合 p-value 門檻
HEDGE_RATIO       = 1.0   # 固定對沖比率（1:1），可根據需要調整
Z_ENTRY           = 2
Z_EXIT            = 0

# 交易執行與資金控管參數
INITIAL_CAPITAL = 10000  # 初始本金
CAPITAL_TRANCHES = math.ceil(TRADING_WINDOW / ROLLING_WINDOW) + 1      # 滾動視窗資金切割份數
TRANSACTION_COST = 0.0029 # 雙邊交易手續費 (0.29%)
MAX_LOSS_PCT      = 5

# 確保快取檔案的上一層資料夾存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。


### 1-3 資料讀取與清理

**清理流程**: 前向填充(最多5天) → 剪除稀疏股票 → 對齊日期

**資料庫結構 (`sp500.db`)**:

| 資料表 | 關鍵欄位 |
|--------|----------|
| `daily_prices` | `ticker, date, adj_close` |
| `tickers` | `ticker, sector` |


In [3]:
def load_data_from_db(db_path, start_date, end_date):
    """Load price and sector data from SQLite."""
    conn = sqlite3.connect(db_path)
    price_queries = [
        (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, close FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'Price rows: {len(prices_df):,}')
                break
        except Exception:
            continue
    if prices_df is None or len(prices_df) == 0:
        tbls = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
        conn.close()
        raise RuntimeError(f"No price table. Tables: {tbls['name'].tolist()}")
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'Sector rows: {len(sector_df):,}')
                break
        except Exception:
            continue
    conn.close()
    if sector_df is None or len(sector_df) == 0:
        print('WARNING: No sector table, using Unknown.')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
        
    # [Mod] Mitigate Survivorship Bias: Check if delisted components exist
    max_date = prices_df['date'].max()
    max_dates = prices_df.groupby('ticker')['date'].max()
    delisted_count = (max_dates < max_date - pd.Timedelta(days=30)).sum()
    if delisted_count < 10:
        import logging
        logging.warning("STRICT RESEARCH LIMITATION: Database lacks historically delisted S&P 500 components. Survivorship Bias is present.")
        
    return prices_df, sector_df



def fix_unknown_sectors(sector_df, use_dynamic=True, save_path=r'data\imputed_sectors.csv'):
    """具備本機快取與全域開關控制的產業補齊模組"""
    # 1. 開關判斷：如果不使用動態補齊，直接原封不動回傳
    if not use_dynamic:
        print("【消融實驗設定】不使用動態產業補齊，維持原始 Unknown 分類作為對照組。")
        return sector_df

    # 確保儲存的目錄 (data\) 存在
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # 2. 快取讀取：如果已經抓過並存檔，直接載入
    if os.path.exists(save_path):
        print(f"從本機快取載入已補齊的產業分類: {save_path}")
        cached_df = pd.read_csv(save_path)
        update_df = cached_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        return sector_df.reset_index()

    # 3. API 抓取：如果沒有快取，執行連線作業
    unknown_mask = sector_df['sector'] == 'Unknown'
    unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
    
    if not unknown_tickers:
        return sector_df

    print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔 Unknown 股票的產業分類...")
    
    yf_logger = logging.getLogger('yfinance')
    original_level = yf_logger.level
    yf_logger.setLevel(logging.CRITICAL) 
    
    fixed_sectors = []
    
    for i, ticker in enumerate(unknown_tickers):
        try:
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            fixed_sectors.append({'ticker': ticker, 'sector': sector})
            time.sleep(0.02) 
        except Exception:
            fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
        if (i + 1) % 50 == 0:
            print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
            
    yf_logger.setLevel(original_level)
    
    # 4. 儲存快取：將剛抓下來的資料存成 CSV，下次就不用再抓了
    fetched_df = pd.DataFrame(fixed_sectors)
    fetched_df.to_csv(save_path, index=False)
    print(f"API 抓取完畢！已將動態產業分類永久儲存至: {save_path}")
    
    # 更新回原本的 DataFrame
    update_df = fetched_df.set_index('ticker')
    sector_df = sector_df.set_index('ticker')
    sector_df.update(update_df)
    sector_df = sector_df.reset_index()
    
    remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
    print(f"補齊完成！剩餘真實無法識別(已下市)的 Unknown 股票數量: {remaining}")
    
    return sector_df

def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """Pivot, forward-fill, drop sparse tickers."""
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    pivot.ffill(limit=5, inplace=True)
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    print(f'Matrix: {len(pivot)} days x {len(pivot.columns)} tickers')

    # [Mod] Integrate Macroeconomic Indicator (VIX)
    import yfinance as yf
    print("Fetching VIX data...")
    try:
        vix_data = yf.download("^VIX", start=pivot.index.min(), end=pivot.index.max() + pd.Timedelta(days=1), progress=False)
        if isinstance(vix_data.columns, pd.MultiIndex):
            vix_close = vix_data['Close'].squeeze()
        else:
            vix_close = vix_data['Close']
        vix_df = pd.DataFrame({'VIX': vix_close})
        vix_df.index = pd.to_datetime(vix_df.index).tz_localize(None)
        vix_aligned = vix_df.reindex(pivot.index).ffill()
        print("VIX feature matrix aligned.")
    except Exception as e:
        print(f"Error fetching VIX: {e}")
        vix_aligned = pd.DataFrame(index=pivot.index, columns=['VIX'])

    return pivot, vix_aligned


In [4]:
# 1. 從資料庫載入原始資料
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)

# 2. 攔截並補齊 Unknown 產業分類 (傳入全域開關與路徑)
sector_info = fix_unknown_sectors(
    sector_info, 
    use_dynamic=USE_DYNAMIC_SECTORS, 
    save_path=IMPUTED_SECTOR_PATH
)

# 3. 執行原有的價格矩陣轉換與 VIX 融合
price_pivot, vix_features = preprocess_prices(prices_raw)
sector_map = sector_info.set_index('ticker')['sector'].to_dict()

# 4. 顯示結果
print(pd.Series(sector_map).value_counts().head(10))

Price rows: 616,467
Sector rows: 843
從本機快取載入已補齊的產業分類: ..\data\imputed_sectors.csv
Matrix: 1008 days x 623 tickers
Fetching VIX data...
VIX feature matrix aligned.
Unknown                   214
Industrials                94
Financials                 76
Information Technology     71
Health Care                60
Consumer Discretionary     48
Consumer Cyclical          37
Real Estate                36
Consumer Staples           36
Energy                     35
Name: count, dtype: int64


### 配對選擇（形成期）

**Engle-Granger 協整檢驗**：
$$P_{A,t} = \alpha + \beta P_{B,t} + \varepsilon_t$$
如果殘差 $\varepsilon_t$ 的 ADF $p < 0.01$，則判定協整。

**SSD 排序**: $\text{SSD}(A,B)=\sum(r_A^{norm}-r_B^{norm})^2$

SSD 越小 → 趨勢越相似 → 優先選入交易期。

**選定流程**: 同產業候選 → 協整檢驗 ($p<0.01$) → SSD由小到大排序 → Top-N


## Z值計算

**線性迴歸模型** - 最小平方法（Least Squares, LS）：

$$y_{1t} = \mu + \gamma \cdot y_{2t} + \epsilon_t$$

**參數定義**：
- $y_{1t}, y_{2t}$：兩檔資產的價格序列
- $\mu$：長期均衡值（截距項）
- $\gamma$：共整合係數/對沖比例（迴歸係數）
- $\epsilon_t$：殘差項（交易的基礎訊號）

**估計方法**：透過最小化誤差平方和
$$\min \sum_{t=1}^T (y_{1t} - (\mu + \gamma y_{2t}))^2$$

得到 $\hat{\mu}$ 與 $\hat{\gamma}$

**Z值計算**：
$$Z_t = \frac{\hat{\epsilon}_t - \text{mean}(\hat{\epsilon})}{\text{std}(\hat{\epsilon})} = \frac{(y_{1t} - \hat{\gamma} y_{2t} - \hat{\mu}) - \text{mean}(\hat{\epsilon})}{\text{std}(\hat{\epsilon})}$$

In [5]:
### 最小平方法（LS）
def compute_ls_parameters(price_a, price_b):
    """
    使用最小平方法估計線性迴歸模型的參數
    
    模型：y₁ₜ = μ + γ·y₂ₜ + εₜ
    
    其中：
    - y₁ₜ, y₂ₜ：兩檔資產的價格序列
    - μ：長期均衡值
    - γ：共整合係數/對沖比例
    - εₜ：殘差項（即Z值）
    
    Returns:
    --------
    dict 包含：
        'mu': 截距項 μ
        'gamma': 迴歸係數 γ
        'residuals': 殘差序列 (可選)
        'r_squared': R² 值
    """
    from scipy import stats as sp_stats
    
    # 清理數據
    valid_idx = ~(np.isnan(price_a) | np.isnan(price_b) | np.isinf(price_a) | np.isinf(price_b))
    pa = price_a[valid_idx]
    pb = price_b[valid_idx]
    
    if len(pa) < 3:
        return None
    
    # OLS 迴歸：y1 ~ y2
    slope, intercept, r_value, p_value, std_err = sp_stats.linregress(pb, pa)
    
    return {
        'mu': intercept,        # 截距 μ
        'gamma': slope,         # 迴歸係數 γ
        'r_squared': r_value**2,
        'p_value': p_value
    }

In [6]:
# ===== Z值計算=====
def execute_zscore(trade_prices, pair, form_prices):
    """
    計算配對的Z值序列
    
    Parameters:
    -----------
    trade_prices : pd.DataFrame
        交易期的價格DataFrame
    pair : dict
        配對物件，包含 ls_mu, ls_gamma 等信息
    form_prices : pd.DataFrame
        形成期的價格DataFrame
    
    Returns:
    --------
    pd.Series : 交易期每個交易日的Z值
              缺失日期填充為NaN
    """
    a, b = pair['stock_a'], pair['stock_b']
    try:
        pa_t, pb_t = trade_prices[a].dropna(), trade_prices[b].dropna()
        pa_f, pb_f = form_prices[a].dropna(), form_prices[b].dropna()
    except KeyError:
        return pd.Series(0.0, index=trade_prices.index)
    
    common = pa_t.index.intersection(pb_t.index)
    fi = pa_f.index.intersection(pb_f.index)
    
    if len(common) < 5 or len(fi) < 5:
        return pd.Series(np.nan, index=trade_prices.index)
    
    # ===== 取得 LS 參數 =====
    ls_mu = pair.get('ls_mu', 0.0)
    ls_gamma = pair.get('ls_gamma', 1.0)
    
    # ===== 計算形成期的殘差統計量 =====
    pa_f_vals = pa_f[fi].values
    pb_f_vals = pb_f[fi].values
    
    residuals_formation = pa_f_vals - ls_gamma * pb_f_vals - ls_mu
    hist_mean = residuals_formation.mean()
    hist_std = residuals_formation.std()
    
    if hist_std < 1e-10:
        return pd.Series(np.nan, index=trade_prices.index)
    
    # ===== 計算交易期的Z值 =====
    pa_t_vals = pa_t[common].values
    pb_t_vals = pb_t[common].values
    residuals_trading = pa_t_vals - ls_gamma * pb_t_vals - ls_mu
    z_values = (residuals_trading - hist_mean) / hist_std
    
    # 構建Z值序列（使用交易期的通用日期索引）
    z_series = pd.Series(z_values, index=common, dtype=float)
    
    return z_series.reindex(trade_prices.index, fill_value=np.nan)


# ===== 交易執行（基於Z值的交易邏輯）=====
def execute_pair_trades(z_series, trade_prices, pair, form_prices,
                        z_entry=Z_ENTRY, z_exit=Z_EXIT,
                        max_loss_pct=MAX_LOSS_PCT, cost=TRANSACTION_COST):
    """
    基於Z值的配對交易執行
    
    Parameters:
    -----------
    z_series : pd.Series
        由 execute_zscore() 生成的Z值序列
    trade_prices : pd.DataFrame
        交易期的價格DataFrame
    pair : dict
        配對物件，包含 ls_mu, ls_gamma 等信息
    form_prices : pd.DataFrame
        形成期的價格DataFrame（用於計算LS參數）
    z_entry : float
        進場Z值門檯
    z_exit : float
        出場Z值門檯
    max_loss_pct : float
        停損比例（%）
    cost : float
        交易成本比例
    
    Returns:
    --------
    pd.Series : 每日百分比損益 (Daily PnL)
    """
    a, b = pair['stock_a'], pair['stock_b']
    try:
        pa_t, pb_t = trade_prices[a].dropna(), trade_prices[b].dropna()
        pa_f, pb_f = form_prices[a].dropna(), form_prices[b].dropna()
    except KeyError:
        return pd.Series(0.0, index=trade_prices.index)
    
    common = pa_t.index.intersection(pb_t.index)
    fi = pa_f.index.intersection(pb_f.index)
    
    if len(common) < 5 or len(fi) < 5:
        return pd.Series(0.0, index=trade_prices.index)
    
    ls_mu = pair.get('ls_mu', 0.0)
    ls_gamma = pair.get('ls_gamma', 1.0)
    
    # ===== 提取Z值序列 =====
    z_values = z_series[common].values
    pa_t_vals = pa_t[common].values
    pb_t_vals = pb_t[common].values
    
    # ===== 狀態機與PnL計算 =====
    pos = 0  # 持倉狀態：1=Long A/Short B, -1=Short A/Long B, 0=無倉
    target_pos = 0
    entry_pa, entry_pb = 0.0, 0.0
    in_cooldown = False
    pnl_d = {}
    
    actual_stop_loss = -abs(max_loss_pct) / 100.0 if max_loss_pct > 0 else max_loss_pct
    
    for i in range(len(common)):
        zi = z_values[i]
        cpa, cpb = pa_t_vals[i], pb_t_vals[i]
        dt = common[i]
        p = 0.0
        
        # 計算現有持倉的每日損益
        if i > 0 and pos != 0:
            prev_pa, prev_pb = pa_t_vals[i-1], pb_t_vals[i-1]
            # Long A / Short B: +A -B
            # Short A / Long B: -A +B
            p += pos * ((cpa - prev_pa) / prev_pa - ls_gamma * (cpb - prev_pb) / prev_pb)
        
        # 倉位轉換
        if target_pos != pos:
            # 交易成本（買賣雙邊）
            p -= 2 * cost * (1 + ls_gamma)
            if target_pos != 0:
                entry_pa, entry_pb = cpa, cpb
            pos = target_pos
        
        # 計算未實現損益（用於停損判斷）
        unrealized_pnl = 0.0
        if pos != 0 and entry_pa > 0 and entry_pb > 0:
            unrealized_pnl = pos * ((cpa - entry_pa) / entry_pa - ls_gamma * (cpb - entry_pb) / entry_pb)
        
        # 狀態轉移邏輯
        if in_cooldown:
            # 冷卻期：等待Z值回到安全區間
            if abs(zi) < z_exit:
                in_cooldown = False
        elif pos == 0:
            # 無倉位：檢查進場訊號
            if zi > z_entry:
                # Z > entry：A相對於B被高估，做空A做多B
                target_pos = -1
            elif zi < -z_entry:
                # Z < -entry：A相對於B被低估，做多A做空B
                target_pos = +1
        else:
            # 有倉位：檢查出場條件
            if unrealized_pnl <= actual_stop_loss:
                # 停損
                target_pos = 0
                in_cooldown = True
            elif (pos == 1 and zi >= -z_exit) or (pos == -1 and zi <= z_exit):
                # 均值回歸：Z值回到安全區間
                target_pos = 0
        
        pnl_d[dt] = p
    
    return pd.Series(pnl_d, dtype=float).reindex(trade_prices.index, fill_value=0.0)

In [7]:
### 三種配對篩選方法的實作

# ================================================================
# 方法 I：SSD - Sum of Squared Differences
# ================================================================

def select_pairs_ssd(price_window, sector_map, top_n=TOP_N_PAIRS,coint_pval=COINT_P_VALUE, sector_neutral=SECTOR_NEUTRAL):
    """
    SSD方法：計算正規化價格曲線間的距離
    流程：協整檢驗 -> SSD排序 -> LS參數估計 -> Top-N
    
    返回的配對物件包含：
    - stock_a, stock_b, sector, coint_p, ssd
    - ls_mu, ls_gamma: LS模型參數
    - historical_residuals: 形成期殘差
    """
    def normalize_prices(prices):
        return prices / prices.iloc[0]
    
    def compute_ssd(na, nb):
        return ((na - nb)**2).sum()
    
    norm = normalize_prices(price_window.dropna(axis=1))
    valid = norm.columns.tolist()
    valid = [t for t in valid if sector_map.get(t, 'Unknown') != 'Unknown']
    
    if sector_neutral:
        groups = {}
        for t in valid:
            sec = sector_map.get(t, 'Unknown')
            groups.setdefault(sec, []).append(t)
        candidates = [(a, b, s) for s, ms in groups.items() if len(ms) >= 2 for a, b in combinations(ms, 2)]
    else:
        candidates = [(a, b, 'All') for a, b in combinations(valid, 2)]
    
    if not candidates:
        return []
    
    min_len = FORMATION_WINDOW * 0.8
    min_len_int = int(min_len)
    
    # 協整檢驗 + SSD計算 + LS參數估計
    results = []
    for a, b, sector in candidates:
        try:
            pa = price_window[a].dropna()
            pb = price_window[b].dropna()
            idx = pa.index.intersection(pb.index)
            if len(idx) >= min_len_int:
                pval = coint(pa[idx].values, pb[idx].values)[1]
                if pval < coint_pval:
                    # SSD計算
                    ssd = compute_ssd(norm[a], norm[b])
                    
                    # LS參數估計（使用原始價格）
                    pa_vals = pa[idx].values
                    pb_vals = pb[idx].values
                    ls_params = compute_ls_parameters(pa_vals, pb_vals)
                    
                    if ls_params:
                        results.append({
                            'stock_a': a,
                            'stock_b': b,
                            'sector': sector,
                            'coint_p': pval,
                            'ssd': ssd,
                            'ls_mu': ls_params['mu'],
                            'ls_gamma': ls_params['gamma'],
                            'r_squared': ls_params['r_squared'],
                            'historical_residuals': pa_vals - ls_params['gamma'] * pb_vals - ls_params['mu']
                        })
        except:
            pass
    
    if not results:
        return []
    
    # 依SSD排序
    df = pd.DataFrame(results).sort_values('ssd')
    return df.head(top_n).to_dict('records')


# ================================================================
# 方法 II：協整法 - Cointegration-Based
# ================================================================

def select_pairs_cointegration(price_window, sector_map, top_n=TOP_N_PAIRS, coint_pval=COINT_P_VALUE, sector_neutral=SECTOR_NEUTRAL):
    """
    協整法：基於統計強度（p-value）篩選
    流程：全量協整檢驗 -> p值排序 -> LS參數估計 -> Top-N
    
    返回的配對物件包含：
    - stock_a, stock_b, sector, coint_p
    - ls_mu, ls_gamma: LS模型參數
    - historical_residuals: 形成期殘差
    - r_squared: 迴歸決定係數
    """
    valid = price_window.columns.tolist()
    valid = [t for t in valid if sector_map.get(t, 'Unknown') != 'Unknown']
    
    if sector_neutral:
        groups = {}
        for t in valid:
            sec = sector_map.get(t, 'Unknown')
            groups.setdefault(sec, []).append(t)
        candidates = [(a, b, s) for s, ms in groups.items() if len(ms) >= 2 for a, b in combinations(ms, 2)]
    else:
        candidates = [(a, b, 'All') for a, b in combinations(valid, 2)]
    
    if not candidates:
        return []
    
    min_len = FORMATION_WINDOW * 0.8
    min_len_int = int(min_len)
    
    results = []
    for a, b, sector in candidates:
        try:
            pa = price_window[a].dropna()
            pb = price_window[b].dropna()
            idx = pa.index.intersection(pb.index)
            if len(idx) >= min_len_int:
                # 雙向協整檢驗
                pval_ab = coint(pa[idx].values, pb[idx].values)[1]
                pval_ba = coint(pb[idx].values, pa[idx].values)[1]
                min_pval = min(pval_ab, pval_ba)
                
                if min_pval < coint_pval:
                    # LS參數估計
                    pa_vals = pa[idx].values
                    pb_vals = pb[idx].values
                    ls_params = compute_ls_parameters(pa_vals, pb_vals)
                    
                    if ls_params:
                        results.append({
                            'stock_a': a,
                            'stock_b': b,
                            'sector': sector,
                            'coint_p': min_pval,
                            'ls_mu': ls_params['mu'],
                            'ls_gamma': ls_params['gamma'],
                            'r_squared': ls_params['r_squared'],
                            'p_value': ls_params['p_value'],
                            'historical_residuals': pa_vals - ls_params['gamma'] * pb_vals - ls_params['mu']
                        })
        except:
            pass
    
    if not results:
        return []
    
    # 依p-value排序（p小優先）
    df = pd.DataFrame(results).sort_values('coint_p')
    return df.head(top_n).to_dict('records')


# ================================================================
# 方法 III：DTW - Dynamic Time Warping
# ================================================================

def select_pairs_dtw(price_window, sector_map, top_n=TOP_N_PAIRS, coint_pval=COINT_P_VALUE, sector_neutral=SECTOR_NEUTRAL):
    """
    DTW方法：動態時間扭曲距離
    流程：協整檢驗 (篩選候選) -> DTW計算 -> DTW排序 -> LS參數估計 -> Top-N
    
    返回的配對物件包含：
    - stock_a, stock_b, sector, dtw_distance
    - ls_mu, ls_gamma: LS模型參數
    - historical_residuals: 形成期殘差
    """
    def dtw_distance(x, y):
        """計算DTW距離"""
        n, m = len(x), len(y)
        dtw_matrix = np.full((n + 1, m + 1), np.inf)
        dtw_matrix[0, 0] = 0
        
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                cost = abs(x[i - 1] - y[j - 1])
                dtw_matrix[i, j] = cost + min(dtw_matrix[i - 1, j],
                                              dtw_matrix[i - 1, j - 1],
                                              dtw_matrix[i, j - 1])
        return dtw_matrix[n, m]
    
    def normalize_returns(prices):
        """對數報酬"""
        return np.diff(np.log(prices.values))
    
    valid = price_window.columns.tolist()
    valid = [t for t in valid if sector_map.get(t, 'Unknown') != 'Unknown']
    
    if sector_neutral:
        groups = {}
        for t in valid:
            sec = sector_map.get(t, 'Unknown')
            groups.setdefault(sec, []).append(t)
        candidates = [(a, b, s) for s, ms in groups.items() if len(ms) >= 2 for a, b in combinations(ms, 2)]
    else:
        candidates = [(a, b, 'All') for a, b in combinations(valid, 2)]
    
    if not candidates:
        return []
    
    min_len = FORMATION_WINDOW * 0.8
    min_len_int = int(min_len)
    
    # 篩選階段：協整檢驗（確保統計有效性）
    coint_passed = []
    for a, b, sector in candidates:
        try:
            pa = price_window[a].dropna()
            pb = price_window[b].dropna()
            idx = pa.index.intersection(pb.index)
            if len(idx) >= min_len_int:
                pval_ab = coint(pa[idx].values, pb[idx].values)[1]
                pval_ba = coint(pb[idx].values, pa[idx].values)[1]
                if min(pval_ab, pval_ba) < coint_pval:
                    coint_passed.append((a, b, sector, idx))
        except:
            pass
    
    if not coint_passed:
        return []
    
    # DTW計算與LS參數估計
    results = []
    for a, b, sector, idx in coint_passed:
        try:
            pa = price_window[a][idx].values
            pb = price_window[b][idx].values
            
            ret_a = normalize_returns(price_window[a][idx])
            ret_b = normalize_returns(price_window[b][idx])
            
            # 對齊長度
            min_len_ret = min(len(ret_a), len(ret_b))
            if min_len_ret > 10:
                dtw_dist = dtw_distance(ret_a[:min_len_ret], ret_b[:min_len_ret])
                
                # LS參數估計
                ls_params = compute_ls_parameters(pa, pb)
                
                if ls_params:
                    results.append({
                        'stock_a': a,
                        'stock_b': b,
                        'sector': sector,
                        'dtw_distance': dtw_dist,
                        'ls_mu': ls_params['mu'],
                        'ls_gamma': ls_params['gamma'],
                        'r_squared': ls_params['r_squared'],
                        'historical_residuals': pa - ls_params['gamma'] * pb - ls_params['mu']
                    })
        except:
            pass
    
    if not results:
        return []
    
    # 依DTW距離排序
    df = pd.DataFrame(results).sort_values('dtw_distance')
    return df.head(top_n).to_dict('records')

In [ ]:
### 6-2.5：優化版回測引擎 + 線程池並行（Jupyter 穩定版）

import time
from concurrent.futures import ThreadPoolExecutor

print("\n" + "="*70)
print("🚀 【優化版】線程池並行回測（Jupyter 穩定版）")
print("="*70)

def run_single_backtest(method_func, method_name):
    """執行單一方法的完整回測（線程模型）"""
    return run_multi_method_backtest(
        price_pivot, sector_map, method_func,
        method_name=method_name,
        top_n=TOP_N_PAIRS
    )

t_total = time.time()
results_all_methods = {}  # 保持與後續代碼兼容

try:
    # ThreadPoolExecutor：線程池（Jupyter 環境穩定）
    # 不用 ProcessPoolExecutor 避免 pickle 序列化問題
    with ThreadPoolExecutor(max_workers=3, thread_name_prefix='backtest') as executor:
        # 提交三個方法的任務
        futures_map = {
            executor.submit(run_single_backtest, select_pairs_ssd, '[方法I] SSD'): 'SSD',
            executor.submit(run_single_backtest, select_pairs_cointegration, '[方法II] 協整法'): 'Cointegration',
            executor.submit(run_single_backtest, select_pairs_dtw, '[方法III] DTW'): 'DTW'
        }
        
        # 依序收集結果
        for future in futures_map:
            try:
                pnl, records = future.result(timeout=3600)  # 最多等待 1 小時
                method_key = futures_map[future]
                elapsed = time.time() - t_total
                
                results_all_methods[method_key] = {
                    'pnl': pnl,
                    'records': records,
                    'time': elapsed
                }
                print(f"✓ {method_key} 完成（累計耗時 {elapsed:.1f}秒）")
            except Exception as e:
                print(f"❌ {futures_map[future]} 執行失敗: {str(e)[:100]}")

except Exception as e:
    print(f"⚠️  線程池執行異常: {str(e)[:200]}")
    print("\n降級為串行模式...")
    
    # 備用方案：串行執行（如線程池失敗）
    methods = [
        (select_pairs_ssd, '[方法I] SSD', 'SSD'),
        (select_pairs_cointegration, '[方法II] 協整法', 'Cointegration'),
        (select_pairs_dtw, '[方法III] DTW', 'DTW')
    ]
    
    for method_func, method_name, key in methods:
        try:
            t_start = time.time()
            pnl, records = run_multi_method_backtest(
                price_pivot, sector_map, method_func,
                method_name=method_name,
                top_n=TOP_N_PAIRS
            )
            elapsed = time.time() - t_start
            results_all_methods[key] = {
                'pnl': pnl,
                'records': records,
                'time': elapsed
            }
            print(f"✓ {key} 完成（耗時 {elapsed:.1f}秒）")
        except Exception as me:
            print(f"❌ {key} 失敗: {str(me)[:100]}")

total_time = time.time() - t_total
print(f"\n⏱️  【總耗時】{total_time:.1f} 秒")
print(f"\n【性能對比】")
for method, data in results_all_methods.items():
    if 'time' in data:
        print(f"  {method}: {data['time']:.1f}秒")


🚀 【優化版本】多進程並行回測開始
⚠️  多進程執行失敗，改用串行模式: A process in the process pool was terminated abruptly while the future was running or pending.

🚀 啟動 [方法I] SSD 回測 | 初始資金: $10,000
   每視窗配置: $1,250.00


In [ ]:
### 6-3：【已基於上一個 cell 執行】結果整理

# results_all_methods 已由前一個 cell 的優化線程池版本產生
# 以下是結果統計

print("\n✓ 回測完成！結果已保存至 results_all_methods")
print(f"  SSD: {results_all_methods['SSD']['time']:.1f}秒")
print(f"  Cointegration: {results_all_methods['Cointegration']['time']:.1f}秒")  
print(f"  DTW: {results_all_methods['DTW']['time']:.1f}秒")

In [ ]:
# 後續分析依賴 results_all_methods
# 該變數已由 6-2.5 cell 生成

In [ ]:
### 6-4：三方法績效對標

# 計算所有方法的績效指標
metrics_list = []
for method_name, result in results_all_methods.items():
    pnl_df = result['pnl']
    metrics = compute_strategy_metrics(pnl_df, INITIAL_CAPITAL, method_name)
    if metrics:
        metrics_list.append(metrics)

metrics_df = pd.DataFrame(metrics_list).set_index('方法')

# 視覺化對標
print('\n' + '='*80)
print('績效對標表（三大配對篩選方法）')
print('='*80)
display(metrics_df.style.background_gradient(cmap='RdYlGn', axis=0, subset=metrics_df.columns[:-2]))

# 依Sharpe排名
print('\n🏆 Sharpe Ratio 排名：')
sharpe_rank = metrics_df['Sharpe'].sort_values(ascending=False)
for i, (method, score) in enumerate(sharpe_rank.items(), 1):
    print(f'  {i}. {method}: {score:.4f}')

# 依CAGR排名
print('\n📈 年化報酬(CAGR)排名：')
cagr_rank = metrics_df['CAGR(%)'].sort_values(ascending=False)
for i, (method, score) in enumerate(cagr_rank.items(), 1):
    print(f'  {i}. {method}: {score:.2f}%')

# 依最大回撤排名（數值越小越好）
print('\n📉 最大回撤(MDD)排名（越小越好）：')
mdd_rank = metrics_df['最大回撤(%)'].sort_values()
for i, (method, score) in enumerate(mdd_rank.items(), 1):
    print(f'  {i}. {method}: {score:.2f}%')


In [ ]:
### 6-5：三方法的 NAV 曲線與回撤對比

# 準備三個方法的報酬序列
returns_comparison = {}
for method_name, result in results_all_methods.items():
    pnl_df = result['pnl']
    portfolio_pnl = pnl_df.sum(axis=1)
    returns_comparison[method_name] = portfolio_pnl / INITIAL_CAPITAL

# 繪製 NAV 曲線與回撤
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']  # 藍、橙、綠
fig_compare = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['NAV曲線對比', '最大回撤對比'],
    vertical_spacing=0.08
)

for idx, (method, ret_series) in enumerate(returns_comparison.items()):
    ret_clean = ret_series.dropna().replace([np.inf, -np.inf], np.nan).dropna()
    nav = (1 + ret_clean).cumprod()
    dd = ((nav - nav.cummax()) / nav.cummax()) * 100
    
    color = COLORS[idx]
    
    # NAV 曲線
    fig_compare.add_trace(
        go.Scatter(
            x=nav.index, y=nav.values,
            name=method,
            line=dict(color=color, width=2.5),
            hovertemplate=f'%{{x|%Y-%m-%d}}<br>NAV: %{{y:.4f}}<extra>{method}</extra>'
        ),
        row=1, col=1
    )
    
    # 回撤
    fig_compare.add_trace(
        go.Scatter(
            x=dd.index, y=dd.values,
            name=method, showlegend=False,
            fill='tozeroy',
            line=dict(color=color, width=1),
            hovertemplate=f'%{{x|%Y-%m-%d}}<br>DD: %{{y:.2f}}%<extra>{method}</extra>'
        ),
        row=2, col=1
    )

fig_compare.update_layout(
    title='三大配對篩選方法績效對比',
    height=700,
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation='h', y=1.02, x=1, xanchor='right', yanchor='bottom')
)
fig_compare.update_yaxes(title_text='NAV', row=1, col=1)
fig_compare.update_yaxes(title_text='回撤 (%)', row=2, col=1)

fig_compare.show()

print('✓ 三方法對比圖已繪製')


In [ ]:
### 6-6：配對選擇差異分析

# 比較三個方法選出的配對集合
print('\n' + '='*80)
print('配對選擇差異分析')
print('='*80)

# 彙整各方法的配對集合
pairs_by_method = {}
for method_name, result in results_all_methods.items():
    records = result['records']
    all_pairs = set()
    for rec in records:
        for pair in rec['pairs']:
            pair_str = f"{pair['stock_a']}-{pair['stock_b']}"
            all_pairs.add(pair_str)
    pairs_by_method[method_name] = all_pairs
    print(f'\n{method_name}：共選出 {len(all_pairs)} 組獨特配對')

# 計算方法間的重疊度
print('\n配對交集統計：')
ssd_pairs = pairs_by_method['SSD']
coint_pairs = pairs_by_method['Cointegration']
dtw_pairs = pairs_by_method['DTW']

overlap_ssd_coint = len(ssd_pairs & coint_pairs)
overlap_ssd_dtw = len(ssd_pairs & dtw_pairs)
overlap_coint_dtw = len(coint_pairs & dtw_pairs)
overlap_all = len(ssd_pairs & coint_pairs & dtw_pairs)

print(f'  SSD ∩ 協整法: {overlap_ssd_coint} 組 ({overlap_ssd_coint/max(len(ssd_pairs),1)*100:.1f}%)')
print(f'  SSD ∩ DTW: {overlap_ssd_dtw} 組 ({overlap_ssd_dtw/max(len(ssd_pairs),1)*100:.1f}%)')
print(f'  協整法 ∩ DTW: {overlap_coint_dtw} 組 ({overlap_coint_dtw/max(len(coint_pairs),1)*100:.1f}%)')
print(f'  三者共同: {overlap_all} 組')

# 方法特異的配對
unique_ssd = ssd_pairs - coint_pairs - dtw_pairs
unique_coint = coint_pairs - ssd_pairs - dtw_pairs
unique_dtw = dtw_pairs - ssd_pairs - coint_pairs

print(f'\n方法專屬配對：')
print(f'  SSD 獨有: {len(unique_ssd)} 組')
print(f'  協整法獨有: {len(unique_coint)} 組')
print(f'  DTW 獨有: {len(unique_dtw)} 組')

# 視窗層級的配對配置統計
print(f'\n視窗層級統計（每個視窗的配對平均數）：')
avg_pairs_ssd = np.mean([len(r['pairs']) for r in records_ssd])
avg_pairs_coint = np.mean([len(r['pairs']) for r in records_coint])
avg_pairs_dtw = np.mean([len(r['pairs']) for r in records_dtw])

print(f'  SSD: {avg_pairs_ssd:.1f} 對/視窗')
print(f'  協整法: {avg_pairs_coint:.1f} 對/視窗')
print(f'  DTW: {avg_pairs_dtw:.1f} 對/視窗')


## 附錄：回測優化詳解

### 1. 為什麼改用 ThreadPoolExecutor？

**原問題**：`ProcessPoolExecutor` 在 Jupyter 中會失敗
```
A process in the process pool was terminated abruptly while the future was running or pending.
```

**根本原因**：
- ProcessPoolExecutor 需要使用 **Pickle** 序列化 DataFrame 等複雜對象跨進程傳輸
- Jupyter Notebook 環境中，全局 `price_pivot` 等變數無法正確被子進程訪問
- Windows 環境下 ProcessPoolExecutor 還有額外的兼容性問題

**ThreadPoolExecutor 解決方案**：
- 線程在同一進程內，**直接共享內存**，無需序列化
- 在 Jupyter 中完全穩定可靠
- 適合 I/O 密集 + 計算任務混合場景

### 2. 當前優化層級

#### 第 1 層：並行策略執行（6-2.5 cell）
```
SSD  ┐
協整法 ├─ 線程池執行（3 個線程同時運行）
DTW   ┘
```
- **加速倍數**：理想情況 3x（實際 ~2.5x 因為線程GIL限制）

#### 第 2 層：配對篩選並行（可選，未啟用）
```
協整檢驗（1000+ 候選對）→ ThreadPoolExecutor（6 工作線程）
```
- 需要修改 `batch_coint_test()` 函數
-** 加速倍數**：4-6x

### 3. 進一步優化建議

#### 推薦：增強配對篩選並行
在 6-2.5 cell 前添加以下代碼：

```python
def select_pairs_ssd_parallel(price_window, sector_map, top_n=TOP_N_PAIRS,
                              coint_pval=COINT_P_VALUE, sector_neutral=SECTOR_NEUTRAL):
    """增加配對篩選內部的協整檢驗並行"""
    # ... [省略前置邏輯] ...
    
    # 並行協整檢驗
    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = [executor.submit(test_coint_pair, a, b, s) for a, b, s in candidates]
        results = [f.result() for f in futures if f.result()]
    
    # ... [後續 SSD 計算] ...
```

#### 參數調整優化
| 參數 | 當前值 | 推薦試驗 | 效果 |
|------|--------|---------|------|
| `TOP_N_PAIRS` | 5 | 2-3 | ↓ 計算量 50-70% |
| `ROLLING_WINDOW` | 20 天 | 30-50 天 | ↓ 視窗數 30-60% |
| `COINT_P_VALUE` | 0.01 | 0.001-0.005 | ↑ 候選配對篩選 |
| `max_workers` | 3 | 6-8 | ↑ 並行度（需評估 CPU） |

#### 時間預估（基于 FAST_TEST_MODE）
| 組態 | SSD | 協整法 | DTW | 總計 |
|------|-----|--------|-----|-----|
| 原版串行 | 15分 | 20分 | 45分 | **80分** |
| 線程池並行(cur) | 15分 | 20分 | 45分 | **45分** ⚡ |
| +配對篩選並行 | 5分 | 8分 | 18分 | **18分** 🚀 |

### 4. 故障排除

**問題 1**：線程池仍然卡住
```
→ 通常是因為某個視窗的配對篩選太困難
→ 解決：在 run_multi_method_backtest() 中添加進度條
```

**問題 2**：內存占用過高
```
→ 原因：所有三個 DataFrame (pnl_ssd, pnl_coint, pnl_dtw) 同時在內存
→ 解決：改為逐個執行並立即計算指標、保存結果
```

**問題 3**：某個方法單獨特別慢（如 DTW）
```
→ DTW 的 O(n²) 複雜度是瓶頸
→ 解決：啟用 dtaidistance 庫（C 實現），或改用 fastdtw 近似算法
→ 代碼：pip install dtaidistance 並在配對篩選中啟用
```

### 5. 使用建議

- ✅ **推薦**：保持當前線程池方案（穩定 + 快速）
- ⚠️  **謹慎**：增加 max_workers 前先測試內存占用
- ❌ **不推薦**：回到 ProcessPoolExecutor（在 Jupyter 中不穩定）